In [1]:
import pandas as pd
import geopandas as gpd
import zipfile
import numpy as np
import matplotlib.pyplot as plt

In [12]:
measles_tx_24_25 = pd.read_csv('../TX_MMR_vax_rate_24_25.csv')

In [13]:
measles_tx_24_25.columns

Index(['Facility Number', 'School Type', 'School District or Name',
       'Facility Address', 'County', 'MMR Vaccination Rate', 'Age Group'],
      dtype='str')

In [ ]:
### Read  zip code and county shapefiles
#texas_districts_shape = gpd.read_file('../shp_files/Current_Districts_2025.shp')
us_zips_shape = gpd.read_file('../shp_files/tl_2025_us_zcta520.shp')
counties_shape = gpd.read_file('../shp_files/tl_2025_48_cousub.shp')

In [ ]:
### Set both shape files to equal-area crs
ea_crs = 'EPSG:5070'
county_ea = counties_shape.to_crs(ea_crs)
zips_ea = us_zips_shape.to_crs(ea_crs)

In [ ]:
### Filter county shape file to only include counties with measeles data
county_ea_tx = county_ea[county_ea['NAME'].isin(list(measles_tx_24_25['County']))]

9       Trinity
27        Milam
30       Jasper
36      Coleman
41       Denton
         ...   
795    Hemphill
813      Burnet
825       Moore
842    Comanche
845    Gonzales
Name: NAME, Length: 61, dtype: str

In [18]:
### Overlap zip codes and counties
zip_county_intersect = gpd.overlay(zips_ea[['ZCTA5CE20', 'geometry']], county_ea_tx[['NAME', 'geometry']], how='intersection', keep_geom_type=False)

In [26]:
zip_county_intersect['area'] = zip_county_intersect.geometry.area
zip_county_intersect = zip_county_intersect.rename(columns={'ZCTA5CE20': 'ZIP', 'NAME': 'County'})

In [27]:
zip_county_intersect

,ZIP,County,geometry,area
0,78016,Moore,GEOMETRYCOLLECTION (POLYGON ((-278946.98 67272...,1.750212e+07
1,75928,Newton,GEOMETRYCOLLECTION (POLYGON ((214170.245 84880...,1.272019e+08
2,78717,Austin,"MULTILINESTRING ((-164887.057 821511.131, -164...",0.000000e+00
3,76201,Denton,"POLYGON ((-108666.797 1129643.451, -108589.102...",1.422503e+07
4,75835,Crockett,GEOMETRYCOLLECTION (POLYGON ((38367.329 924723...,5.824918e+08
...,...,...,...,...
717,75126,Terrell,"MULTILINESTRING ((-35450.791 1082581.444, -354...",0.000000e+00
718,75160,Terrell,"POLYGON ((-34983.444 1074322.267, -34983.711 1...",1.592291e+08
719,78754,Austin,GEOMETRYCOLLECTION (POLYGON ((-161846.704 8086...,1.610522e+07
720,76641,Dawson,"MULTILINESTRING ((-68334.331 987011.121, -6849...",0.000000e+00


In [46]:
zip_to_county = zip_county_intersect.copy()
zip_to_county['overlap_area'] = zip_to_county.geometry.area

In [47]:
zip_to_county = zip_to_county.assign(total_in_group = zip_to_county.groupby('County')['overlap_area'].transform('sum'))

In [45]:
zip_to_county['percent_in_group'] = zip_to_county.groupby('County')['ZIP'].transform(lambda x: x['overlap_area']/x['total_in_group'])

KeyError: 'overlap_area'

In [48]:
zip_to_county

,ZIP,County,geometry,area,overlap_area,total_in_group
0,78016,Moore,GEOMETRYCOLLECTION (POLYGON ((-278946.98 67272...,1.750212e+07,1.750212e+07,4.711003e+08
1,75928,Newton,GEOMETRYCOLLECTION (POLYGON ((214170.245 84880...,1.272019e+08,1.272019e+08,8.604435e+08
2,78717,Austin,"MULTILINESTRING ((-164887.057 821511.131, -164...",0.000000e+00,0.000000e+00,1.226865e+09
3,76201,Denton,"POLYGON ((-108666.797 1129643.451, -108589.102...",1.422503e+07,1.422503e+07,2.141072e+08
4,75835,Crockett,GEOMETRYCOLLECTION (POLYGON ((38367.329 924723...,5.824918e+08,5.824918e+08,7.237819e+08
...,...,...,...,...,...,...
717,75126,Terrell,"MULTILINESTRING ((-35450.791 1082581.444, -354...",0.000000e+00,0.000000e+00,4.412055e+08
718,75160,Terrell,"POLYGON ((-34983.444 1074322.267, -34983.711 1...",1.592291e+08,1.592291e+08,4.412055e+08
719,78754,Austin,GEOMETRYCOLLECTION (POLYGON ((-161846.704 8086...,1.610522e+07,1.610522e+07,1.226865e+09
720,76641,Dawson,"MULTILINESTRING ((-68334.331 987011.121, -6849...",0.000000e+00,0.000000e+00,6.172339e+08
